In [91]:
!pip install lightgbm tensorflow scikit-learn pandas numpy

In [92]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [93]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
print("Path:", path)

Using Colab cache for faster access to the 'm5-forecasting-accuracy' dataset.
Path: /kaggle/input/m5-forecasting-accuracy


In [94]:
import os

os.listdir(path)

['m5-forecasting-accuracy']

In [95]:
os.listdir(f"{path}/m5-forecasting-accuracy")

['calendar.csv',
 'sample_submission.csv',
 'sell_prices.csv',
 'sales_train_validation.csv',
 'sales_train_evaluation.csv']

In [96]:
import pandas as pd

In [97]:
sales = pd.read_csv(f"{path}/m5-forecasting-accuracy/sales_train_validation.csv")
calendar = pd.read_csv(f"{path}/m5-forecasting-accuracy/calendar.csv")
prices = pd.read_csv(f"{path}/m5-forecasting-accuracy/sell_prices.csv")

In [98]:
import os

dataset_path = os.path.join(path, "m5-forecasting-accuracy")

sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [99]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [100]:
print(sales.shape)
print(calendar.shape)
print(prices.shape)

(30490, 1919)
(1969, 14)
(6841121, 4)


In [101]:
print(path)
os.listdir(path)

/kaggle/input/m5-forecasting-accuracy


['m5-forecasting-accuracy']

In [102]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [103]:
sales = sales[
    (sales['state_id'].isin(['CA','TX','WI'])) &
    (sales['cat_id'].isin(['HOBBIES','HOUSEHOLD','FOODS']))
]

sales = sales.sample(frac=0.5, random_state=42)


day_cols = [c for c in sales.columns if 'd_' in c]
half_days = day_cols[:len(day_cols)//2]

sales = sales[['item_id','dept_id','cat_id','store_id','state_id'] + half_days]

In [104]:
sales_long = sales.melt(
    id_vars=['item_id','dept_id','cat_id','store_id','state_id'],
    var_name='d',
    value_name='sales'
)

In [105]:
sales_long = sales_long.merge(
    calendar[['d', 'wm_yr_wk', 'month']],
    on='d',
    how='left'
)

In [106]:
sales_long = sales_long.merge(
    prices,
    on=['store_id','item_id','wm_yr_wk'],
    how='left'
)

In [113]:
df = df.sort_values(
    ['cat_id', 'state_id', 'wm_yr_wk']
)

for lag in [1, 2, 3, 4, 8, 12]:
    df[f'lag_{lag}'] = (
        df.groupby(['cat_id', 'state_id'])['revenue']
          .shift(lag)
    )

df['rolling_mean_4'] = (
    df.groupby(['cat_id', 'state_id'])['revenue']
      .transform(lambda x: x.shift(1).rolling(4).mean())
)

df['rolling_mean_12'] = (
    df.groupby(['cat_id', 'state_id'])['revenue']
      .transform(lambda x: x.shift(1).rolling(12).mean())
)

df = df.dropna()

# Reset index after preprocessing
df = df.reset_index(drop=True)

# Create chronological position within each time series
df['group_pos'] = df.groupby(
    ['cat_id', 'state_id']
).cumcount()

# Number of observations in each series
df['group_size'] = df.groupby(
    ['cat_id', 'state_id']
)['revenue'].transform('size')

# 80% chronological split for EACH series
df['split_point'] = (
    df['group_size'] * 0.8
).astype(int)

# Training and validation masks
train_mask = df['group_pos'] < df['split_point']
val_mask = df['group_pos'] >= df['split_point']

print("Training rows:", train_mask.sum())
print("Validation rows:", val_mask.sum())

Training rows: 1098
Validation rows: 279


In [114]:
features = [
    'lag_1',
    'lag_2',
    'lag_3',
    'lag_4',
    'lag_8',
    'lag_12',
    'rolling_mean_4',
    'rolling_mean_12',
    'month'
]

X = df[features]
y = df['revenue']

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]

y_train = y.loc[train_mask]
y_val = y.loc[val_mask]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (1098, 9)
X_val: (279, 9)


In [83]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_val   = X.iloc[split:]

y_train = y.iloc[:split]
y_val   = y.iloc[split:]

In [84]:
model_lgb = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    random_state=42
)

model_lgb.fit(
    X_train,
    y_train
)

pred_lgb = model_lgb.predict(X_val)

print("LightGBM predictions:", pred_lgb.shape)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000372 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2053
[LightGBM] [Info] Number of data points in the train set: 1101, number of used features: 9
[LightGBM] [Info] Start training from score 29372.714280


In [85]:
lookback = 10

X_lstm = []
y_lstm = []
target_indices = []

# Create sequences separately for each category-state combination
for (cat, state), group in df.groupby(['cat_id', 'state_id']):

    group = group.sort_values('wm_yr_wk')

    values = group['revenue'].values
    indices = group.index.values

    for i in range(lookback, len(values)):

        X_lstm.append(
            values[i-lookback:i]
        )

        y_lstm.append(
            values[i]
        )

        # Store the original dataframe index
        target_indices.append(
            indices[i]
        )

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)
target_indices = np.array(target_indices)

X_lstm = X_lstm.reshape(
    X_lstm.shape[0],
    X_lstm.shape[1],
    1
)

print("LSTM X shape:", X_lstm.shape)
print("LSTM y shape:", y_lstm.shape)

In [86]:
# Determine whether each LSTM target belongs to train or validation
lstm_train_mask = train_mask.loc[target_indices].values
lstm_val_mask = val_mask.loc[target_indices].values

X_lstm_train = X_lstm[lstm_train_mask]
X_lstm_val = X_lstm[lstm_val_mask]

y_lstm_train = y_lstm[lstm_train_mask]
y_lstm_val = y_lstm[lstm_val_mask]

lstm_val_indices = target_indices[lstm_val_mask]

print("LSTM training:", X_lstm_train.shape)
print("LSTM validation:", X_lstm_val.shape)

In [87]:
model_lstm = Sequential([
    LSTM(
        64,
        input_shape=(lookback, 1)
    ),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model_lstm.compile(
    optimizer="adam",
    loss="mse"
)

history = model_lstm.fit(
    X_lstm_train,
    y_lstm_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 1448106752.0000 - val_loss: 1190688384.0000
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1447899520.0000 - val_loss: 1190414208.0000
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1447612416.0000 - val_loss: 1190057216.0000
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1447236352.0000 - val_loss: 1189595520.0000
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1446759168.0000 - val_loss: 1189018496.0000
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1446171392.0000 - val_loss: 1188341888.0000
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1445516288.0000 - val_loss: 1187572992.0000
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1444749696.0000 - val_loss: 1186696704.0000
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1443934848.0000 - val_loss: 1185724544.0000
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1442971136.0000 - val_loss: 1

In [88]:
pred_lstm = model_lstm.predict(
    X_lstm_val,
    verbose=0
).flatten()

print("LSTM predictions:", pred_lstm.shape)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


In [90]:
# Validation rows used by LightGBM
lgb_val_indices = df.index[val_mask]

# Convert LightGBM predictions into a Series indexed by dataframe row
pred_lgb_series = pd.Series(
    pred_lgb,
    index=lgb_val_indices
)

# Convert LSTM predictions into a Series indexed by target row
pred_lstm_series = pd.Series(
    pred_lstm,
    index=lstm_val_indices
)

# Find rows predicted by BOTH models
common_indices = (
    pred_lgb_series.index
    .intersection(pred_lstm_series.index)
)

# Align everything using the same target rows
actual = df.loc[
    common_indices,
    'revenue'
].values

pred_lgb_aligned = pred_lgb_series.loc[
    common_indices
].values

pred_lstm_aligned = pred_lstm_series.loc[
    common_indices
].values

print("Aligned actual:", actual.shape)
print("Aligned LightGBM:", pred_lgb_aligned.shape)
print("Aligned LSTM:", pred_lstm_aligned.shape)

NameError: name 'val_lgb' is not defined

In [ ]:
best_weight = None
best_rmse = float("inf")

for w in np.arange(0, 1.01, 0.01):

    hybrid_pred = (
        w * pred_lgb_aligned
        + (1 - w) * pred_lstm_aligned
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            hybrid_pred
        )
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_weight = w

print("Best LightGBM weight:", best_weight)
print("Best LSTM weight:", 1 - best_weight)
print("Best RMSE:", best_rmse)

In [ ]:
hybrid_pred = (
    best_weight * pred_lgb_aligned
    + (1 - best_weight) * pred_lstm_aligned
)

In [ ]:
mae = mean_absolute_error(
    actual,
    hybrid_pred
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        hybrid_pred
    )
)

# Avoid division by zero
non_zero_mask = actual != 0

mape = np.mean(
    np.abs(
        (actual[non_zero_mask] - hybrid_pred[non_zero_mask])
        / actual[non_zero_mask]
    )
) * 100

print("Hybrid MAE :", mae)
print("Hybrid RMSE:", rmse)
print("Hybrid MAPE:", mape)